# Tool Use with AG2 Agents

This notebook shows how to give AG2 agents the ability to call Python functions as tools. Combined with OpenAI's function calling, agents can fetch real data, run calculations, and take actions — not just generate text.

**What you'll learn:**
- Registering Python functions as agent tools
- How AG2's decorator pattern maps to OpenAI function calling
- Building an agent that fetches live data and reasons over it

**Prerequisites:**
- OpenAI API key
- Python 3.10+

In [ ]:
%pip install "ag2[openai]>=0.11.4,<1.0" -q

In [ ]:
import os
import json
from datetime import datetime
from autogen import AssistantAgent, UserProxyAgent, LLMConfig

llm_config = LLMConfig({
    "model": "gpt-4o-mini",
    "api_key": os.environ["OPENAI_API_KEY"],
    "api_type": "openai",
})

## Defining Tools

AG2 uses a decorator pattern for tool registration:
- `@user_proxy.register_for_execution()` — tells the user proxy to execute the function when called
- `@assistant.register_for_llm(description="...")` — tells the assistant this function is available, mapping it to OpenAI's function calling format

This two-step registration separates *who decides to call a tool* (the LLM-powered assistant) from *who executes it* (the user proxy).

In [ ]:
assistant = AssistantAgent(
    name="Analyst",
    system_message=(
        "You are a data analyst. Use the available tools to look up "
        "information and perform calculations. Present results clearly. "
        "Reply TERMINATE when done."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config=False,
)


# --- Tool definitions ---

@user_proxy.register_for_execution()
@assistant.register_for_llm(description="Get current date and time in ISO format")
def get_current_time() -> str:
    return datetime.now().isoformat()


@user_proxy.register_for_execution()
@assistant.register_for_llm(
    description="Calculate compound interest. annual_rate_pct is the annual interest rate as a percentage (e.g. 10 for 10%). Returns the final amount."
)
def compound_interest(
    principal: float,
    annual_rate_pct: float,
    years: int,
    compounds_per_year: int = 12,
) -> str:
    rate = annual_rate_pct / 100
    amount = principal * (1 + rate / compounds_per_year) ** (compounds_per_year * years)
    return json.dumps({
        "principal": principal,
        "rate": f"{rate:.2%}",
        "years": years,
        "final_amount": round(amount, 2),
        "total_interest": round(amount - principal, 2),
    })


@user_proxy.register_for_execution()
@assistant.register_for_llm(
    description="Look up the current price of a stock ticker. Returns a simulated price for demo purposes."
)
def get_stock_price(ticker: str) -> str:
    # Simulated prices for demonstration
    prices = {
        "AAPL": 227.50, "MSFT": 445.20, "GOOGL": 178.90,
        "AMZN": 205.30, "NVDA": 131.80, "META": 595.40,
    }
    price = prices.get(ticker.upper())
    if price is None:
        return json.dumps({"error": f"Ticker '{ticker}' not found"})
    return json.dumps({"ticker": ticker.upper(), "price": price, "currency": "USD"})

## Running the Tool-Using Agent

When the assistant decides a tool is needed, it generates an OpenAI function call. The user proxy intercepts this, executes the registered function, and returns the result. The assistant then reasons over the output.

In [ ]:
user_proxy.run(
    assistant,
    message=(
        "I have $50,000 to invest. Look up the current prices for AAPL, MSFT, "
        "and NVDA. Then calculate how much I'd have after 5 years if I invested "
        "equally across all three, assuming a 10% annual return compounded monthly."
    ),
).process()

## How It Works Under the Hood

1. The assistant receives the user message and decides which tools to call
2. AG2 maps the `@register_for_llm` decorators to OpenAI's `tools` parameter in the API call
3. When the model returns a `tool_calls` response, the user proxy finds the matching `@register_for_execution` function and runs it
4. The tool result is sent back to the assistant for further reasoning

This pattern works with any Python function — API calls, database queries, file operations, or calculations.

## Next Steps

- **Async Tools**: Register async functions for I/O-bound operations
- **GroupChat + Tools**: Combine multi-agent orchestration with tool use
- **Code Execution**: Let agents write and execute arbitrary Python code
- **Learn more**: [ag2.ai](https://ag2.ai) | [AG2 Docs](https://docs.ag2.ai) | [GitHub](https://github.com/ag2ai/ag2)